In [1]:
import random
import numpy as np
from deap import base, creator, tools, algorithms

# ---------- Setup (unchanged) ----------
random.seed(42)

NDIM = 2
LOW  = [-5.0, -5.0]
UP   = [5.0, 5.0]

POP_SIZE = 20
NGEN     = 50
CXPB     = 0.9
MUTPB    = 1.0
ETA_C    = 15.0
ETA_M    = 20.0
INDPB    = 1.0 / NDIM

def rosenbrock(ind):
    x1, x2 = ind
    return (100.0 * (x2 - x1**2)**2 + (1.0 - x1)**2,)

# ---------- DEAP classes & toolbox ----------
if not hasattr(creator, "FitnessMin"):
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_x1", random.uniform, LOW[0], UP[0])
toolbox.register("attr_x2", random.uniform, LOW[1], UP[1])
toolbox.register("individual",
    tools.initCycle, creator.Individual,
    (toolbox.attr_x1, toolbox.attr_x2), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", rosenbrock)
toolbox.register("select", tools.selTournament, tournsize=2)
toolbox.register("mate", tools.cxSimulatedBinaryBounded, low=LOW, up=UP, eta=ETA_C)
toolbox.register("mutate", tools.mutPolynomialBounded, low=LOW, up=UP, eta=ETA_M, indpb=INDPB)

# ---------- Initial population ----------
pop = toolbox.population(n=POP_SIZE)
for ind in pop:
    ind.fitness.values = toolbox.evaluate(ind)

# ---------- Compact GA loop (μ + λ) ----------
print("=" * 70)
print("Initial Population")
print("=" * 70)
for i, ind in enumerate(pop, 1):
    print(f"{i:>3} | {ind[0]:>10.4f} | {ind[1]:>10.4f} | {ind.fitness.values[0]:>14.4f}")

for gen in range(1, NGEN + 1):
    # 1) Select parents, then apply crossover & mutation (independent, no sum limit)
    parents = toolbox.select(pop, POP_SIZE)       # tournament selection
    offspring = algorithms.varAnd(parents, toolbox, cxpb=CXPB, mutpb=MUTPB)

    # 2) Evaluate new individuals
    invalid = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = toolbox.map(toolbox.evaluate, invalid)
    for ind, fit in zip(invalid, fitnesses):
        ind.fitness.values = fit

    # 3) Survival: (μ + λ) – keep best from parents + offspring
    pop = tools.selBest(pop + offspring, POP_SIZE)
    #pop = tools.selBest(offspring, POP_SIZE)

    # 4) Print generation status (exact same style as your original)
    best = tools.selBest(pop, 1)[0]
    print(f"\nGeneration {gen}:")
    print(f"Best solution = [{best[0]:.4f}, {best[1]:.4f}]")
    print(f"Best fitness  = {best.fitness.values[0]:.6f}")

# ---------- Final result ----------
best = tools.selBest(pop, 1)[0]
print("\n" + "=" * 70)
print("Final Best Solution")
print("=" * 70)
print(f"x = [{best[0]:.6f}, {best[1]:.6f}]")
print(f"f(x) = {best.fitness.values[0]:.6f}")

Initial Population
  1 |     1.3943 |    -4.7499 |      4480.9526
  2 |    -2.2497 |    -2.7679 |      6139.9996
  3 |     2.3647 |     1.7670 |      1464.8244
  4 |     3.9218 |    -4.1306 |     38076.8119
  5 |    -0.7808 |    -4.7020 |      2824.5317
  6 |    -2.8136 |     0.0536 |      6197.0728
  7 |    -4.7346 |    -3.0116 |     64693.4538
  8 |     1.4988 |     0.4494 |       323.2128
  9 |    -2.7956 |     0.8927 |      4806.7671
 10 |     3.0943 |    -4.9350 |     21057.6217
 11 |     3.0582 |     1.9814 |      5437.6177
 12 |    -1.5975 |    -3.4452 |      3603.3815
 13 |     4.5721 |    -1.6341 |     50810.8603
 14 |    -4.0725 |    -4.0328 |     42537.7005
 15 |     3.4749 |     1.0373 |     12189.8105
 16 |     3.0713 |     2.2973 |      5095.7688
 17 |     0.3623 |     4.7312 |      2116.3241
 18 |    -1.2147 |     0.5204 |        96.1040
 19 |     3.2940 |     1.1852 |      9347.5401
 20 |     3.6171 |     0.7735 |     15159.6385

Generation 1:
Best solution = [0.7757, 0